# Analizando el Dataset: Restaurant Orders

## Importando el dataset

El primer paso que haremos es comprender el dataset que vamos a importar en cuanto a los datos que posee y cómo podremos utilizarlo para satisfacer el requerimiento del tp.

El dataset a utilizar es el de [restaurant-orders](https://www.kaggle.com/datasets/haseebindata/restaurant-orders), un Dataset que contiene las órdenes de comidas de un restaurante registrando el tiempo en el que ocurrieron.

In [8]:
#Importamos el Dataset directo desde Kaggle con su respectiva API
import kagglehub


path = kagglehub.dataset_download("haseebindata/restaurant-orders", output_dir="./dataset",force_download=True)

print("Path to dataset files:", path)

100%|██████████████████████████████████████| 11.7k/11.7k [00:00<00:00, 16.5MB/s]

Extracting files...
Path to dataset files: ./dataset


In [1]:
#Observamos cómo vienen los datos crudos
import pandas as pd

datasetRaw = pd.read_csv("F:/Simulación/dataset-tp/dataset/restaurant_orders.csv")
datasetRaw

,Order ID,Customer Name,Food Item,Category,Quantity,Price,Payment Method,Order Time
0,2268,Mary Vega DDS,Pasta,Main,5,16.52,Cash,2025-02-02 14:28:41
1,3082,Brandon Myers,Brownie,Dessert,4,17.27,Debit Card,2025-06-08 10:57:47
2,3160,Margaret Wells,Pasta,Main,1,3.37,Credit Card,2025-03-04 07:41:41
3,1272,Michael Matthews,Pasta,Main,5,2.20,Online Payment,2025-05-15 12:43:45
4,9447,Connor Williams,Soup,Starter,1,12.23,Cash,2025-03-15 14:25:56
...,...,...,...,...,...,...,...,...
495,6323,Alyssa Anthony,Pizza,Main,1,21.31,Cash,2025-01-15 19:21:02
496,9836,Jerry Pineda,Soup,Starter,3,15.99,Debit Card,2025-07-15 15:00:19
497,1202,Brandy Smith,Pasta,Main,2,8.54,Credit Card,2025-08-03 23:47:28
498,7876,Ivan Haynes,Soup,Starter,5,20.54,Credit Card,2025-07-23 08:10:06


In [10]:
datasetRaw.sort_values(by='Order ID')

,Order ID,Customer Name,Food Item,Category,Quantity,Price,Payment Method,Order Time
288,1055,David Gutierrez,Ice Cream,Dessert,5,3.97,Online Payment,2025-01-08 22:56:09
420,1057,Hannah Brown,Soup,Starter,5,5.12,Credit Card,2025-04-08 03:59:14
403,1066,Melissa Duncan,Fries,Starter,2,22.97,Debit Card,2025-07-06 08:10:00
401,1084,Eric Nguyen,Pizza,Main,4,22.41,Online Payment,2025-03-24 01:48:12
30,1088,Carol Lynch,Ice Cream,Dessert,1,14.82,Cash,2025-04-12 10:21:40
...,...,...,...,...,...,...,...,...
454,9927,Gary Phillips,Pasta,Main,2,17.57,Debit Card,2025-03-24 20:55:14
480,9936,Debbie Castillo,Salad,Starter,2,15.76,Credit Card,2025-02-24 01:19:37
129,9963,Jonathan Turner,Pasta,Main,4,3.07,Debit Card,2025-02-05 21:59:57
358,9969,Monica Hill,Fries,Starter,3,9.42,Debit Card,2025-06-07 09:10:16


In [13]:
#Veamos si las órdenes siguen un rastro de visita
duplicados = datasetRaw[datasetRaw.duplicated(subset='Order ID', keep=False)]

In [14]:
duplicados = datasetRaw[datasetRaw.duplicated(subset='Customer Name', keep=False)]

## Curando los datos

Para Esta sección, transformaremos este dataset para extrapolarlo al caso de uso que necesitamos.
Como requerimos hacer una simulación de colas, interpretaremos los datos de la siguiente manera:

* Haremos una única cola donde 'Order Time' será usado como el 'Tiempo de arribo'
* Añadiremos en nuestro algoritmo una variable de control que abarcará el número de mesas (Suponiendo que no se puede juntarlas)
* El tiempo de atención lo calcularemos sumando el tiempo de las órdenes las cuales les pondremos un

### Calculando tiempo de atención por comidas

In [8]:
#Empezamos viendo cuáles son las diferentes comidas que se ofrecen en el restaurante
comidas = datasetRaw['Food Item'].unique()
comidas

<StringArray>
[    'Pasta',   'Brownie',      'Soup',      'Cake',    'Burger', 'Ice Cream',
     'Fries',     'Pizza',     'Salad']
Length: 9, dtype: str

In [9]:
comidas_dict = {
    'Pasta': 15,
    'Brownie': 10,
    'Soup': 10,
    'Cake': 15,
    'Burger': 25,
    'Ice Cream': 8,
    'Fries': 12,
    'Pizza': 20,
    'Salad': 5,
}
    

### Calculando tiempo de atención por orden

In [21]:
datasetCurado = datasetRaw
datasetCurado['TPLL'] = datasetRaw['Order Time']
datasetCurado['Comensales'] = datasetRaw['Quantity']
datasetCurado['Ingreso'] = datasetRaw['Price'] * datasetRaw['Quantity']
datasetCurado['TA'] = datasetRaw['Food Item'].map(comidas_dict) * datasetRaw['Quantity']

datasetCurado = datasetCurado[["Order ID","TPLL", "Comensales", "Ingreso", "TA"]]

datasetCurado

,Order ID,TPLL,Comensales,Ingreso,TA
0,2268,2025-02-02 14:28:41,5,82.60,75
1,3082,2025-06-08 10:57:47,4,69.08,40
2,3160,2025-03-04 07:41:41,1,3.37,15
3,1272,2025-05-15 12:43:45,5,11.00,75
4,9447,2025-03-15 14:25:56,1,12.23,10
...,...,...,...,...,...
495,6323,2025-01-15 19:21:02,1,21.31,20
496,9836,2025-07-15 15:00:19,3,47.97,30
497,1202,2025-08-03 23:47:28,2,17.08,30
498,7876,2025-07-23 08:10:06,5,102.70,50
